# CPU Wave Solver Benchmark

This notebook measures the CPU-only performance of the sonar wave simulation kernel.

It times only the numerical simulation step:

```text
previous/current pressure grids -> next pressure grid
```

The benchmark excludes Matplotlib display updates, keyboard input, mouse input, and Jupyter UI overhead. Fixed pulse firing steps are used so the workload includes source injection while remaining repeatable.


In [ ]:
import time

import numpy as np


In [ ]:
# simulation space
WIDTH, HEIGHT = 320, 240
BORDER = 50

# emission
C = 0.4
K = C * C
frequency = [0.01]
CYCLES = 1
pulse_duration = [int(CYCLES / frequency[0])]

# object
shapes = ["circle", "square", "ellipse"]

# initialisation
source_position = [80, 120]
object_position = [200, 120]
object_radius = [20]
object_shape = [0]

gain = [0.8]
pulse_fired_time = [None]

# benchmark settings
WARMUP_STEPS = 100
MEASURED_STEPS = 1000
fire_steps = [0, 500]


In [ ]:
# grid
yy, xx = np.mgrid[0:HEIGHT, 0:WIDTH]

# damping near the boundary to reduce reflections
damping = np.ones((HEIGHT, WIDTH), np.float32)
for i in range(BORDER):
    v = np.sin(0.5 * np.pi * (i + 1) / BORDER) ** 2
    damping[i, :] = np.minimum(damping[i, :], v)
    damping[HEIGHT - 1 - i, :] = np.minimum(damping[HEIGHT - 1 - i, :], v)
    damping[:, i] = np.minimum(damping[:, i], v)
    damping[:, WIDTH - 1 - i] = np.minimum(damping[:, WIDTH - 1 - i], v)


In [ ]:
def make_object_mask(centre_x, centre_y, radius, shape):
    if shape == "circle":
        return ((xx - centre_x) ** 2 + (yy - centre_y) ** 2) <= radius**2
    elif shape == "square":
        return (np.abs(xx - centre_x) <= radius) & (np.abs(yy - centre_y) <= radius)
    elif shape == "ellipse":
        return (
            ((xx - centre_x) ** 2 / radius**2)
            + ((yy - centre_y) ** 2 / (radius * 0.5) ** 2)
        ) <= 1.0

    raise ValueError(f"unknown shape: {shape}")


In [ ]:
source_stencil = np.array(
    [
        [0.05, 0.10, 0.05],
        [0.10, 1.00, 0.10],
        [0.05, 0.10, 0.05],
    ],
    dtype=np.float32,
)
source_stencil /= source_stencil.sum()


In [ ]:
def fire_pulse(timestep):
    pulse_fired_time[0] = timestep


def add_source_pulse(next_wave, timestep):
    if pulse_fired_time[0] is None:
        return

    age = timestep - pulse_fired_time[0]
    if age >= pulse_duration[0]:
        pulse_fired_time[0] = None
        return

    amplitude = gain[0] * np.sin(2 * np.pi * frequency[0] * age)
    x0 = max(source_position[0] - 1, 0)
    x1 = min(source_position[0] + 2, WIDTH)
    y0 = max(source_position[1] - 1, 0)
    y1 = min(source_position[1] + 2, HEIGHT)
    sx0 = x0 - (source_position[0] - 1)
    sx1 = 3 - ((source_position[0] + 2) - x1)
    sy0 = y0 - (source_position[1] - 1)
    sy1 = 3 - ((source_position[1] + 2) - y1)
    next_wave[y0:y1, x0:x1] += amplitude * source_stencil[sy0:sy1, sx0:sx1]


In [ ]:
def simulation_step(previous_wave, current_wave, object_mask, timestep):
    next_wave = np.empty_like(current_wave)
    centre = current_wave[1:-1, 1:-1]

    up = np.where(object_mask[:-2, 1:-1], centre, current_wave[:-2, 1:-1])
    down = np.where(object_mask[2:, 1:-1], centre, current_wave[2:, 1:-1])
    left = np.where(object_mask[1:-1, :-2], centre, current_wave[1:-1, :-2])
    right = np.where(object_mask[1:-1, 2:], centre, current_wave[1:-1, 2:])
    laplacian = up + down + left + right - 4.0 * centre

    next_wave[1:-1, 1:-1] = 0.0
    fluid = ~object_mask[1:-1, 1:-1]
    next_wave[1:-1, 1:-1][fluid] = (
        2 * centre[fluid]
        - previous_wave[1:-1, 1:-1][fluid]
        + K * laplacian[fluid]
    )

    next_wave[0, :] = next_wave[1, :]
    next_wave[-1, :] = next_wave[-2, :]
    next_wave[:, 0] = next_wave[:, 1]
    next_wave[:, -1] = next_wave[:, -2]

    next_wave *= damping
    current_wave *= damping
    next_wave[object_mask] = 0.0
    current_wave[object_mask] = 0.0

    add_source_pulse(next_wave, timestep)
    return current_wave, next_wave


In [ ]:
def reset_benchmark_state():
    pulse_fired_time[0] = None


def validate_fire_steps(steps):
    if isinstance(steps, int):
        steps = [steps]

    clean_steps = sorted(set(int(step) for step in steps))
    previous = None
    for current in clean_steps:
        if current < 0:
            raise ValueError("fire steps must be non-negative")
        if previous is not None and current - previous < pulse_duration[0]:
            raise ValueError(
                "fire steps must be at least one pulse duration apart "
                f"({pulse_duration[0]} timesteps)"
            )
        previous = current
    return clean_steps


In [ ]:
def print_results(warmup_steps, measured_steps, fire_steps, object_mask, current_wave, kernel_times):
    kernel_times = np.asarray(kernel_times, dtype=np.float64)
    total_kernel_time = float(np.sum(kernel_times))
    kernel_avg = total_kernel_time / measured_steps
    kernel_median = float(np.median(kernel_times))
    kernel_min = float(np.min(kernel_times))
    kernel_max = float(np.max(kernel_times))
    kernel_std = float(np.std(kernel_times))

    interior_cells = (WIDTH - 2) * (HEIGHT - 2)
    fluid_cells = int(np.count_nonzero(~object_mask[1:-1, 1:-1]))
    steps_per_second = measured_steps / total_kernel_time
    cell_updates_per_second = fluid_cells * steps_per_second
    fire_steps_text = ", ".join(str(step) for step in fire_steps)

    print("CPU baseline benchmark:")
    print("workload:")
    print(f"  grid                     : {WIDTH} x {HEIGHT}")
    print(f"  measured steps           : {measured_steps:,} (+{warmup_steps:,} warmup)")
    print(f"  fluid cells updated      : {fluid_cells:,} of {interior_cells:,}")
    print(f"  object                   : {shapes[object_shape[0]]}, radius {object_radius[0]}px")
    print(f"  source position          : {tuple(source_position)}")
    print(f"  frequency / gain         : {frequency[0]:.4f} / {gain[0]:.2f}")
    print(f"  pulses fired at timesteps: {fire_steps_text}")
    print()
    print("performance:")
    print(f"  total timed kernel time  : {total_kernel_time:.6f} s")
    print(f"  average step time        : {kernel_avg * 1e3:.6f} ms")
    print(f"  median step time         : {kernel_median * 1e3:.6f} ms")
    print(f"  min / max step time      : {kernel_min * 1e3:.6f} / {kernel_max * 1e3:.6f} ms")
    print(f"  step time std dev        : {kernel_std * 1e3:.6f} ms")
    print(f"  simulation throughput    : {steps_per_second:,.2f} steps/sec")
    print(f"  cell update throughput   : {cell_updates_per_second / 1e6:,.2f} million cells/sec")
    print()
    print("sanity check:")
    print(f"  final checksum           : {float(np.sum(current_wave)):.9f}")
    print(f"  final max |pressure|     : {float(np.max(np.abs(current_wave))):.9f}")

    return {
        "fluid_cells": fluid_cells,
        "average_step_time_ms": kernel_avg * 1e3,
        "steps_per_second": steps_per_second,
        "million_cell_updates_per_second": cell_updates_per_second / 1e6,
        "final_checksum": float(np.sum(current_wave)),
        "final_max_pressure": float(np.max(np.abs(current_wave))),
    }


In [ ]:
def run_benchmark(warmup_steps=0, measured_steps=5000, fire_steps=None):
    if measured_steps <= 0:
        raise ValueError("measured_steps must be greater than 0")
    if fire_steps is None:
        fire_steps = [0]

    fire_steps = validate_fire_steps(fire_steps)
    fire_step_set = set(fire_steps)

    object_mask = make_object_mask(
        object_position[0],
        object_position[1],
        object_radius[0],
        shapes[object_shape[0]],
    )

    reset_benchmark_state()
    previous_wave = np.zeros((HEIGHT, WIDTH), np.float32)
    current_wave = np.zeros((HEIGHT, WIDTH), np.float32)

    for timestep in range(warmup_steps):
        if timestep in fire_step_set:
            fire_pulse(timestep)

        previous_wave, current_wave = simulation_step(
            previous_wave,
            current_wave,
            object_mask,
            timestep,
        )

    reset_benchmark_state()
    previous_wave = np.zeros((HEIGHT, WIDTH), np.float32)
    current_wave = np.zeros((HEIGHT, WIDTH), np.float32)
    kernel_times = []

    for timestep in range(measured_steps):
        if timestep in fire_step_set:
            fire_pulse(timestep)

        kernel_start = time.perf_counter()
        previous_wave, current_wave = simulation_step(
            previous_wave,
            current_wave,
            object_mask,
            timestep,
        )
        kernel_end = time.perf_counter()

        kernel_times.append(kernel_end - kernel_start)

    return print_results(
        warmup_steps,
        measured_steps,
        fire_steps,
        object_mask,
        current_wave,
        kernel_times,
    )


## Benchmark

Run this cell to measure the CPU wave-solver baseline. The fixed firing schedule keeps the workload repeatable while still including source pulse injection.


In [ ]:
results = run_benchmark(
    warmup_steps=WARMUP_STEPS,
    measured_steps=MEASURED_STEPS,
    fire_steps=fire_steps,
)
results
